In [101]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import keras as kr
import seaborn as sb
import sklearn
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sklearn.set_config(transform_output="pandas")

In [102]:
class OutlierRemover(BaseEstimator, TransformerMixin):
    def __init__(self, factor=3.0):
        self.factor = factor
        self.mean_ = None
        self.std_ = None

    def fit(self, X, y=None):
        X = X.copy()
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        return self

    def transform(self, X):
        X = X.copy()
        std_safe = np.where(self.std_ == 0, 1, self.std_)
        z_scores = (X - self.mean_) / std_safe
        mask = np.abs(z_scores) > self.factor
        X[mask] = np.nan
        return X
class NoRemover(BaseEstimator, TransformerMixin):
    def __init__(self):
        return

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        l=[]
        for c in X.columns:
            if "no" in c:
                l.append(c)
        X=X.drop(columns=l)
        return X
text_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(sparse_output=False)),
    ('no_remover', NoRemover())
])
number_pipeline = Pipeline([
    #('outlier', OutlierRemover(factor=3.0)),
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])
no_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])

In [103]:
def preprocessing(raw_df):
    raw_df=raw_df.drop(columns=['Unnamed: 0'])
    ct = ColumnTransformer([
        ("tx", text_pipeline, ["price_range","blue","four_g","wifi"]),
        ("num", number_pipeline, ["battery_power","clock_speed","fc","int_memory","m_dep","mobile_wt","n_cores","pc","px_height","px_width","ram","sc_h","sc_w","talk_time"]),
        ("no", no_pipeline, ["dual_sim","three_g","touch_screen"])
        ])
    df_t=ct.fit_transform(raw_df)
    df_r=df_t.rename(columns={
       'tx__price_range_high cost':'pr_high',
       'tx__price_range_low cost':'pr_low',
       'tx__price_range_medium cost':'pr_medium',
       'tx__price_range_very high cost':'pr_very_high',
       'tx__blue_yes':'blue',
       'tx__four_g_yes':'4g',
       'tx__wifi_wifi':'wifi',
       'num__battery_power':'battery_power',
       'num__clock_speed':'clock_speed',
       'num__fc':'fc',
       'num__int_memory':'int_memory',
       'num__m_dep':'m_dep',
       'num__mobile_wt':'mobile_wt',
       'num__n_cores':'n_cores',
       'num__pc':'pc',
       'num__px_height':'px_height',
       'num__px_width':'px_width',
       'num__ram':'ram',
       'num__sc_h':'sc_h',
       'num__sc_w':'sc_w',
       'num__talk_time':'talk_time',
       'no__dual_sim':'dual_sim',
       'no__three_g':'3g',
       'no__touch_screen':'touch_screen'
    })
    return df_r
    
    

In [104]:
def display_history(history):
    sb.lineplot(history.history)
    plt.grid()
    plt.xlabel("Epoch")
    plt.show()

In [105]:
def random_split(df):
    et=['pr_low','pr_medium','pr_high','pr_very_high']
    X=df.drop(columns=et)
    y=df[et]
    X_train, X_other, y_train, y_other = train_test_split(X, y, test_size=0.3)
    X_valid, X_test, y_valid, y_test = train_test_split(X_other, y_other, test_size=0.5)
    return X_train, y_train, X_valid, y_valid, X_test, y_test

In [106]:
def full_training(df,model,disp=True):
    X_train, y_train, X_valid, y_valid, X_test, y_test=random_split(df)
    Adam = kr.optimizers.Adam(learning_rate=0.001)
    r_lr=kr.callbacks.ReduceLROnPlateau(patience=1, monitor="accuracy",factor=0.7)
    e_s=kr.callbacks.EarlyStopping(patience=20, monitor="val_accuracy")
    model.compile(optimizer=Adam,loss="categorical_crossentropy",metrics =['accuracy'])
    history=model.fit(X_train,y_train,epochs=100,validation_data=[X_valid,y_valid],callbacks=[r_lr,e_s])
    if(disp):display_history(history)
    loss,acc=model.evaluate(X_test,y_test)
    print(100*acc)
    return 100*acc

In [107]:
def reliable_test(df,f_model,n):
    Acc=[]
    for i in range(n):
        model=f_model()
        Acc.append(full_training(df,model,False))
    avg=sum(Acc)/n
    print("Values:",Acc)
    print("Max:\t",max(Acc))
    print("Min:\t",min(Acc))
    print("Avg:\t",avg)
    return avg

In [108]:
read_csv=pd.read_csv("dataset.csv")
df=pd.DataFrame(read_csv)
df_r=preprocessing(df)
df_r

,pr_high,pr_low,pr_medium,pr_very_high,blue,4g,wifi,battery_power,clock_speed,fc,...,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,dual_sim,3g,touch_screen
0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.908331,0.818308,-0.759254,...,-1.309865,-1.408836,-1.150465,0.397885,-0.789575,0.286772,1.490569,-1.018254,-1.790676,-1.003624
1,1.0,0.0,0.0,0.0,1.0,1.0,0.0,-0.500100,-1.254559,-0.989421,...,-0.649386,0.580254,1.685934,0.474026,1.111300,-0.636529,-0.744837,0.982073,0.558448,0.996389
2,1.0,0.0,0.0,0.0,1.0,1.0,0.0,-1.544623,-1.254559,-0.529087,...,-0.649386,1.384880,1.059716,0.448027,-0.314357,-0.867355,-0.372269,0.982073,0.558448,0.996389
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.426031,1.184108,-0.989421,...,-0.154026,1.279244,1.220875,0.602167,0.873690,0.517598,0.000298,-1.018254,0.558448,-1.003624
4,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.324394,-0.401025,2.002751,...,0.671573,1.261264,-0.100629,-0.658810,-1.027185,-0.867355,0.745434,-1.018254,0.558448,0.996389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1930,0.0,1.0,0.0,0.0,1.0,1.0,0.0,-1.017800,-1.254559,-0.989421,...,0.671573,1.292730,1.460311,-1.348726,0.160862,-0.405704,1.490569,0.982073,0.558448,0.996389
1931,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.652803,1.306042,-0.989421,...,-1.144745,0.602729,1.632981,-0.082178,-0.314357,0.979248,0.931718,0.982073,0.558448,0.996389
1932,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.529650,-0.766825,-0.759254,...,-1.144745,0.497094,0.866325,0.869591,-0.789575,-1.098180,-1.117405,0.982073,0.558448,0.996389
1933,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.619683,-0.766825,-0.068753,...,-0.814506,-0.698608,-1.348460,-1.162087,1.348909,0.979248,1.490569,-1.018254,0.558448,0.996389


In [ ]:
def new_model():
    #"""
    #96.18
    model = kr.Sequential([
        kr.layers.Input((20,)),
        kr.layers.Dense(170, activation="softplus"),
        kr.layers.Dense(140, activation="softplus"),
        kr.layers.Dense(110, activation="softplus"),
        kr.layers.Dense(80, activation="softplus"),
        kr.layers.Dense(50, activation="tanh"),
        kr.layers.Dense(4, activation="softmax")
    ])
    #"""
    """
    #96.28
    model = kr.Sequential([
        kr.layers.Input((20,)),
        kr.layers.Dense(800, activation="softplus"),
        kr.layers.Dense(400, activation="softplus"),
        kr.layers.Dense(200, activation="softplus"),
        kr.layers.Dense(100, activation="tanh"),
        kr.layers.Dense(4, activation="softmax")
    ])
    """
    return model

In [110]:
reliable_test(df_r,new_model,10)

Epoch 1/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.2578 - loss: 1.4381 - val_accuracy: 0.2655 - val_loss: 1.3575 - learning_rate: 0.0010
Epoch 2/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4527 - loss: 1.2606 - val_accuracy: 0.6448 - val_loss: 0.9519 - learning_rate: 0.0010
Epoch 3/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7895 - loss: 0.6541 - val_accuracy: 0.9138 - val_loss: 0.4273 - learning_rate: 0.0010
Epoch 4/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8929 - loss: 0.3812 - val_accuracy: 0.8483 - val_loss: 0.3976 - learning_rate: 0.0010
Epoch 5/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8944 - loss: 0.3205 - val_accuracy: 0.9000 - val_loss: 0.3254 - learning_rate: 0.0010
Epoch 6/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9350 - loss: 0.2303 - val_accuracy: 0.8552 - val_loss: 0.3493 - learning_rate: 0.0010
Epoch 7/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9092 - loss: 0.2633 - val_ac

96.18556678295135